# Embedded AI Final Project

#### 20223265 조해영, 20214904 조성

0. 환경 설정

In [6]:
import json
import platform
import sys

import torch
import torchvision

# ROCm(AMD Radeon) 빌드의 PyTorch도 torch.cuda API를 그대로 쓰므로 device 문자열은 "cuda"이다.
if torch.version.hip:
    GPU_BACKEND = f"ROCm/HIP {torch.version.hip} (AMD)"
elif torch.version.cuda:
    GPU_BACKEND = f"CUDA {torch.version.cuda} (NVIDIA)"
else:
    GPU_BACKEND = "none (CPU-only PyTorch build)"

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("GPU backend:", GPU_BACKEND)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_name = gpu_props.name
    gpu_arch = getattr(gpu_props, "gcnArchName", None) or f"sm_{gpu_props.major}{gpu_props.minor}"
    print(f"GPU: {gpu_name} ({gpu_arch}, {gpu_props.total_memory / 1024**3:.1f} GiB)")
else:
    DEVICE = "cpu"
    gpu_name = gpu_arch = None
    print("[warn] No GPU detected; running on CPU.")
    if not torch.version.hip and not torch.version.cuda:
        print(
            "       This PyTorch build is CPU-only. For an AMD Radeon RX 6800 (gfx1030) on Windows:\n"
            '       pip install --index-url https://stable.repo.amd.com/rocm/whl-next/ '
            '"torch[device-gfx1030]" "torchvision[device-gfx1030]"'
        )

ENV_INFO = {
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "gpu_backend": GPU_BACKEND,
    "device": DEVICE,
    "gpu_name": gpu_name,
    "gpu_arch": gpu_arch,
    "platform": platform.platform(),
}
print()
print(json.dumps(ENV_INFO, indent=2, ensure_ascii=False))

python: 3.13.9
torch: 2.14.0+cpu
torchvision: 0.29.0+cpu
GPU backend: none (CPU-only PyTorch build)
GPU available: False
[warn] No GPU detected; running on CPU.
       This PyTorch build is CPU-only. For an AMD Radeon RX 6800 (gfx1030) on Windows:
       pip install --index-url https://stable.repo.amd.com/rocm/whl-next/ "torch[device-gfx1030]" "torchvision[device-gfx1030]"

{
  "python_version": "3.13.9",
  "torch_version": "2.14.0+cpu",
  "torchvision_version": "0.29.0+cpu",
  "gpu_backend": "none (CPU-only PyTorch build)",
  "device": "cpu",
  "gpu_name": null,
  "gpu_arch": null,
  "platform": "Windows-11-10.0.26200-SP0"
}


### 1. 모델 호출

Classification을 위한 ViT를 호출한다

In [7]:
import timm
from timm.data import create_transform, resolve_model_data_config

MODEL_NAME = "vit_base_patch16_224"
NUM_CLASSES = 1000


def load_model(model_name=MODEL_NAME, device="cpu"):
    model = timm.create_model(model_name, num_classes=NUM_CLASSES, pretrained=True)
    model.to(device)
    model.eval()
    return model


def build_eval_transform(model):
    data_config = resolve_model_data_config(model)
    return create_transform(**data_config, is_training=False), data_config


model = load_model(device=DEVICE)
eval_transform, data_config = build_eval_transform(model)

print("model:", MODEL_NAME)
print("device:", DEVICE)
print("preprocessing:", data_config)

model: vit_base_patch16_224
device: cpu
preprocessing: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'crop_pct': 0.9, 'crop_mode': 'center'}


In [8]:
total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
state_dict_size_bytes = sum(
    tensor.numel() * tensor.element_size() for tensor in model.state_dict().values()
)
state_dict_size_mib = state_dict_size_bytes / (1024 ** 2)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"State dict tensor size: {state_dict_size_mib:.2f} MiB ({state_dict_size_bytes:,} bytes)")

Total parameters: 86,567,656
Trainable parameters: 86,567,656
State dict tensor size: 330.23 MiB (346,270,624 bytes)


### ImageNet calibration / test 데이터

In [9]:
import re
import shutil
import tarfile
import tempfile
from pathlib import Path

IMAGENET_ARCHIVE = Path("./val_images.tar.gz")
IMAGENET_ROOT = Path("./data/imagenet1k/val")
NUM_CLASSES = 1000


def ensure_imagenet_prepared(archive_path, dataset_root):
    if dataset_root.is_dir():
        print(f"Using existing ImageNet files: {dataset_root.resolve()}")
        return
    if not archive_path.is_file():
        raise FileNotFoundError(f"ImageNet archive was not found: {archive_path.resolve()}")

    dataset_root.parent.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive_path.resolve()}...")
    with tempfile.TemporaryDirectory(dir=dataset_root.parent) as temp_dir:
        extract_dir = Path(temp_dir) / "payload"
        extract_dir.mkdir()
        with tarfile.open(archive_path, "r:*") as archive:
            def members_with_progress():
                for count, member in enumerate(archive, start=1):
                    if count % 1000 == 0:
                        print(f"Extracting archive entries: {count:,}", flush=True)
                    yield member

            archive.extractall(extract_dir, members=members_with_progress(), filter="data")

        candidates = [
            directory for directory in [extract_dir, *extract_dir.rglob("*")]
            if directory.is_dir()
            and sum(1 for child in directory.iterdir() if child.is_dir()) == NUM_CLASSES
        ]
        if len(candidates) == 1:
            shutil.move(str(candidates[0]), str(dataset_root))
        elif not candidates:
            images = [path for path in extract_dir.rglob("*") if path.suffix.lower() in {".jpg", ".jpeg", ".png"} and path.is_file()]
            labeled_images = [(path, re.search(r"_(n\d{8})$", path.stem)) for path in images]
            if not labeled_images or any(match is None for _, match in labeled_images):
                raise ValueError("Expected class folders or image filenames ending in _nXXXXXXXX.")
            if len({match.group(1) for _, match in labeled_images}) != NUM_CLASSES:
                raise ValueError("Expected images from 1,000 ImageNet classes.")
            organized_dir = Path(temp_dir) / "organized"
            for count, (path, match) in enumerate(labeled_images, start=1):
                class_dir = organized_dir / match.group(1)
                class_dir.mkdir(parents=True, exist_ok=True)
                destination = class_dir / path.name
                if destination.exists():
                    raise ValueError(f"Duplicate image filename: {path.name}")
                shutil.move(str(path), str(destination))
                if count % 1000 == 0:
                    print(f"Organizing images: {count:,}/{len(labeled_images):,}", flush=True)
            shutil.move(str(organized_dir), str(dataset_root))
        else:
            raise ValueError("Found multiple directories containing 1,000 class folders.")

    print(f"ImageNet validation files: {dataset_root.resolve()}")


ensure_imagenet_prepared(IMAGENET_ARCHIVE, IMAGENET_ROOT)

Extracting C:\Users\82105\3-2\임베디드 AI\Embedded-AI-Final-Project\val_images.tar.gz...
Extracting archive entries: 1,000
Extracting archive entries: 2,000
Extracting archive entries: 3,000
Extracting archive entries: 4,000
Extracting archive entries: 5,000
Extracting archive entries: 6,000
Extracting archive entries: 7,000
Extracting archive entries: 8,000
Extracting archive entries: 9,000
Extracting archive entries: 10,000
Extracting archive entries: 11,000
Extracting archive entries: 12,000
Extracting archive entries: 13,000
Extracting archive entries: 14,000
Extracting archive entries: 15,000
Extracting archive entries: 16,000
Extracting archive entries: 17,000
Extracting archive entries: 18,000
Extracting archive entries: 19,000
Extracting archive entries: 20,000
Extracting archive entries: 21,000
Extracting archive entries: 22,000
Extracting archive entries: 23,000
Extracting archive entries: 24,000
Extracting archive entries: 25,000
Extracting archive entries: 26,000
Extracting arc

In [10]:
import re
from pathlib import Path

from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

IMAGENET_ROOT = Path("./data/imagenet1k/val")
CALIBRATION_RATIO = 0.2
BATCH_SIZE = 32
NUM_WORKERS = 0  # Reliable in notebooks on Windows.
SEED = 42
NUM_CLASSES = 1000

if not IMAGENET_ROOT.is_dir():
    raise FileNotFoundError(f"ImageNet folder was not found: {IMAGENET_ROOT.resolve()}")


class ImageNetClassImageFolder(ImageFolder):
    def find_classes(self, directory):
        classes = [entry.name for entry in Path(directory).iterdir() if entry.is_dir()]
        if len(classes) != NUM_CLASSES:
            raise ValueError(f"Expected 1,000 class folders, found {len(classes)} under {directory}.")
        if all(class_name.isdigit() for class_name in classes):
            classes.sort(key=int)
            if classes != [str(class_id) for class_id in range(NUM_CLASSES)]:
                raise ValueError("Numeric class folders must be IDs from 0 to 999.")
            return classes, {class_name: int(class_name) for class_name in classes}
        if all(re.fullmatch(r"n\d{8}", class_name) for class_name in classes):
            classes.sort()
            return classes, {class_name: class_id for class_id, class_name in enumerate(classes)}
        raise ValueError("Class folders must be ImageNet synsets (nXXXXXXXX) or numeric IDs 0-999.")


def split_calibration_test(dataset, calibration_ratio=CALIBRATION_RATIO, seed=SEED):
    if not 0 < calibration_ratio < 1:
        raise ValueError("Calibration ratio must be between 0 and 1.")
    rng = torch.Generator().manual_seed(seed)
    class_indices = {class_id: [] for class_id in range(NUM_CLASSES)}
    for index, class_id in enumerate(dataset.targets):
        class_indices[class_id].append(index)

    calibration_indices = []
    val_indices = []
    calibration_counts = []
    val_counts = []
    for class_id, indices in class_indices.items():
        if len(indices) < 2:
            raise ValueError(f"Class {class_id} needs at least 2 images to split into both datasets.")
        shuffled = torch.randperm(len(indices), generator=rng).tolist()
        split_at = round(len(indices) * calibration_ratio)
        split_at = min(max(split_at, 1), len(indices) - 1)
        calibration_indices.extend(indices[position] for position in shuffled[:split_at])
        val_indices.extend(indices[position] for position in shuffled[split_at:])
        calibration_counts.append(split_at)
        val_counts.append(len(indices) - split_at)

    return Subset(dataset, calibration_indices), Subset(dataset, val_indices), calibration_counts, val_counts


dataset = ImageNetClassImageFolder(IMAGENET_ROOT, transform=eval_transform)
calibration_dataset, val_dataset, calibration_counts, val_counts = split_calibration_test(dataset)

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE == "cuda",
}
calibration_loader = DataLoader(calibration_dataset, shuffle=False, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)

print(f"Dataset: {IMAGENET_ROOT.resolve()}")
print(f"ImageNet classes: {len(dataset.classes)}")
print(f"Calibration images: {len(calibration_dataset)}")
print(f"Test images: {len(val_dataset)}")
print(f"Per-class calibration counts: {sorted(set(calibration_counts))}")
print(f"Per-class test counts: {sorted(set(val_counts))}")
print(f"Image transform: {eval_transform}")

Dataset: C:\Users\82105\3-2\임베디드 AI\Embedded-AI-Final-Project\data\imagenet1k\val
ImageNet classes: 1000
Calibration images: 10000
Test images: 40000
Per-class calibration counts: [10]
Per-class test counts: [40]
Image transform: Compose(
    Resize(size=248, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.5000, 0.5000, 0.5000]), std=tensor([0.5000, 0.5000, 0.5000]))
)


### 3. Baseline 모델 실행

In [11]:
model.eval()
correct = 0
total = 0
class_correct = torch.zeros(NUM_CLASSES, dtype=torch.int64)
class_total = torch.zeros(NUM_CLASSES, dtype=torch.int64)

with torch.inference_mode():
    for images, labels in val_loader:
        images = images.to(DEVICE, non_blocking=DEVICE == "cuda")
        labels = labels.to(DEVICE, non_blocking=DEVICE == "cuda")

        predictions = model(images).argmax(dim=1)
        matches = predictions == labels
        correct += matches.sum().item()
        total += labels.size(0)
        labels_cpu = labels.cpu()
        class_total += torch.bincount(labels_cpu, minlength=NUM_CLASSES)
        class_correct += torch.bincount(labels_cpu[matches.cpu()], minlength=NUM_CLASSES)

if total == 0:
    raise ValueError("Validation set is empty.")

val_accuracy = correct / total
print(f"Validation accuracy: {val_accuracy:.2%} ({correct}/{total})")

class_results = [
    (class_id, class_correct[class_id].item(), class_total[class_id].item())
    for class_id in range(NUM_CLASSES)
    if class_total[class_id].item() > 0
]
for heading, ranked_classes in (
    ("Top 10 classes", sorted(class_results, key=lambda result: (-result[1] / result[2], result[0]))[:10]),
    ("Bottom 10 classes", sorted(class_results, key=lambda result: (result[1] / result[2], result[0]))[:10]),
):
    print(f"\n{heading}:")
    for class_id, class_hits, class_count in ranked_classes:
        print(f"  {dataset.classes[class_id]} (ID {class_id}): {class_hits / class_count:.2%} ({class_hits}/{class_count})")

KeyboardInterrupt: 